In [ ]:
import mediapipe as mp
import os
import numpy as np

from sklearn.metrics import multilabel_confusion_matrix, accuracy_score
import tensorflow as tf
from tensorflow import keras

from matplotlib import pyplot as plt
import time
import mediapipe as mp
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical

2026-02-24 16:00:37.657847: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
2026-02-24 16:00:37.773520: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-02-24 16:00:40.653775: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [ ]:
DATA_PATH = os.path.join('Data') 

# Actions that we try to detect
actions = np.array(["HI","THANKS","FISH","COW","PLANE","CYCLING"])


no_sequences = 45


sequence_length = 20

In [ ]:
label_map = {label:num for num, label in enumerate(actions)}
sequences, labels = [], []
for action in actions:
    for sequence in range(no_sequences):
        window = []
        for frame_num in range(sequence_length):
            res = np.load(os.path.join(DATA_PATH, action, str(sequence), "{}.npy".format(frame_num)))
            window.append(res)
        sequences.append(window)
        labels.append(label_map[action])

In [ ]:
X = np.array(sequences)
y = to_categorical(labels).astype(int)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

In [ ]:
import h5py, json, shutil, os

# Patch isl6.h5 to remove 'time_major' from LSTM config (Keras 3 compatibility)
src = 'isl6.h5'
dst = 'isl6_patched.h5'
shutil.copy2(src, dst)

def remove_time_major(obj):
    if isinstance(obj, dict):
        obj.pop('time_major', None)
        for v in obj.values():
            remove_time_major(v)
    elif isinstance(obj, list):
        for item in obj:
            remove_time_major(item)

with h5py.File(dst, 'r+') as f:
    cfg_bytes = f.attrs['model_config']
    cfg = json.loads(cfg_bytes)
    remove_time_major(cfg)
    f.attrs['model_config'] = json.dumps(cfg)

print("Patched model saved to", dst)


Patched model saved to isl6_patched.h5


In [ ]:
model=keras.models.load_model('isl6_patched.h5')


In [ ]:
yhat = model.predict(X_test)
ytrue = np.argmax(y_test, axis=1).tolist()
yhat = np.argmax(yhat, axis=1).tolist()

3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 401ms/step


In [ ]:
multilabel_confusion_matrix(ytrue, yhat)

array([[[65,  1],
        [ 0, 15]],

       [[69,  0],
        [ 0, 12]],

       [[66,  1],
        [ 0, 14]],

       [[69,  0],
        [ 0, 12]],

       [[64,  0],
        [ 1, 16]],

       [[70,  0],
        [ 1, 10]]])

In [ ]:
accuracy_score(ytrue, yhat)

0.9753086419753086

In [ ]:
from sklearn.metrics import precision_score, recall_score
precision_score(ytrue, yhat,average='macro')

0.9784722222222223

In [ ]:
recall_score(ytrue, yhat,average='macro')

0.9750445632798574

In [ ]:
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['categorical_accuracy'])
loss, accuracy = model.evaluate(X_test, y_test)
print('Test Loss: {}, Test Accuracy: {}'.format(loss, accuracy))


3/3 ━━━━━━━━━━━━━━━━━━━━ 2s 55ms/step - categorical_accuracy: 0.9753 - loss: 0.1343
Test Loss: 0.1342756152153015, Test Accuracy: 0.9753086566925049
